# Consensus Principle Selection

In [1]:
from pathlib import Path
from typing import Dict, List, Tuple

import time
import sys
import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

In [2]:
DEDUPLICATED_DATA_PATH = Path(r"C:\Users\laras\OneDrive\Dokumente\duke_classes\icai_project\Principle-Elicitation\deduplication\intermediate_principle_clusters_k=1371.json")

TOP_K_PRINCIPLES = 5
K_SUMMARY = 5

TRESHOLD = "k=1371"

SUMMARIZED_DATA_PATH = f"cluster_summarized_{TRESHOLD}_fair_clustering_deduplicated.json"


# Load environment variables from current directory .env and override existing ones
load_dotenv(".env", override=True)

# load api key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI(api_key=api_key)

In [3]:
def summarize_cluster(cluster, prompt, summary_prompt):
    num_retries = 0
    max_retries = 5
    while True:
        try:
            response = client.chat.completions.create(
                model="gpt-4.1-2025-04-14",
                messages=[
                    {"role": "system", "content": summary_prompt},
                    {"role": "user", "content": prompt}
                ],
            )
            summary = response.choices[0].message.content.strip()
            return({
                "cluster_id": cluster["cluster_id"],
                "summarized_principle": summary,
                "original_principles": cluster["principles"]
            })
            break
        except Exception as e:
            print(f"Error: {e}. Retrying in 5 seconds...")
            num_retries += 1
            if num_retries >= max_retries:
                print("Max retries reached. Exiting.")
                sys.exit(1)
            time.sleep(5)

In [4]:
def embed_texts(texts):
    single_input = isinstance(texts, str)
    if single_input:
        texts = [texts]

    batch_size = 256  
    all_vectors = []
    max_retries = 5

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        num_retries = 0
        while True:
            try:
                resp = client.embeddings.create(
                    model="text-embedding-3-small",
                    input=batch,
                )
                all_vectors.extend([item.embedding for item in resp.data])
                break
            except Exception as e:
                num_retries += 1
                print(f"Error: {e}. Retrying in 5 seconds...")
                if num_retries >= max_retries:
                    print("Max retries reached. Exiting.")
                    sys.exit(1)
                time.sleep(5)

    arr = np.array(all_vectors, dtype=float)

    return arr

In [5]:
def extract_principle_strings(cluster_item):
    out = []
    for p in cluster_item.get("principles", []):
        if isinstance(p, (list, tuple)) and len(p) > 0:
            s = (p[0] or "").strip()
        else:
            s = (p or "").strip()
        if s:
            out.append(s)
    return out

In [6]:
with open('principle_summary_prompt.txt', 'r', encoding="utf-8") as f:
    summary_prompt = f.read()

In [7]:
with DEDUPLICATED_DATA_PATH.open("r", encoding="utf-8") as f:
    clusters_output = json.load(f)

all_principles = []
cluster_offsets = []
for cluster in clusters_output:
    principles = [p[0].strip() for p in cluster.get("principles", []) if isinstance(p, (list, tuple)) and p and p[0] and str(p[0]).strip()]
    cluster_offsets.append((len(all_principles), len(all_principles) + len(principles)))
    all_principles.extend(principles)

all_embeddings = embed_texts(all_principles)

summarized_principles = []

for cluster, (a, b) in tqdm(list(zip(clusters_output, cluster_offsets)), total=len(clusters_output)):
    principles = all_principles[a:b]
    if not principles:
        continue

    cluster_embeddings = all_embeddings[a:b]
    centroid = np.mean(cluster_embeddings, axis=0)

    distances = np.linalg.norm(cluster_embeddings - centroid, axis=1)
    closest_indices = np.argsort(distances)[:K_SUMMARY]
    closest_principles = [principles[i] for i in closest_indices]

    principles_text = "\n".join(f"{i+1}. {p}" for i, p in enumerate(closest_principles))
    prompt = f"\n\nPrinciples:\n{principles_text}\n\nSummarized Principle:"

    summarized_principles.append(summarize_cluster(cluster, prompt, summary_prompt))

with open(SUMMARIZED_DATA_PATH, "w", encoding="utf-8") as f:
    json.dump(summarized_principles, f, indent=4, ensure_ascii=False)


100%|██████████████████████████████████████████████████████████████████████████████████| 49/49 [00:30<00:00,  1.58it/s]


In [8]:
summaries = []
summary_cluster_ids = []
originals = []

for idx, item in enumerate(summarized_principles):
    cluster_id = item.get("cluster_id")

    summarized_principle = item.get("summarized_principle").strip()
    principle_list = item.get("original_principles", [])

    if summarized_principle:
        summaries.append(summarized_principle)
        summary_cluster_ids.append(cluster_id)

    for p in principle_list:
        p = (p[0] if isinstance(p, (list, tuple)) and p else p)
        p = (p or "").strip()
        if p:
            originals.append(p)

print(f"#summaries: {len(summaries)}")
print(f"#originals: {len(originals)}")


#summaries: 49
#originals: 1129


In [9]:
emb_orig = embed_texts(originals)  
emb_summaries = embed_texts(summaries)  

N, d = emb_orig.shape if emb_orig.size else (0, 0)
M, d_s = emb_summaries.shape if emb_summaries.size else (0, 0)

print(f"emb_orig shape: {emb_orig.shape}")
print(f"emb_summaries shape: {emb_summaries.shape}")

emb_orig shape: (1129, 1536)
emb_summaries shape: (49, 1536)


In [10]:
global_msd = []  

for j in range(M):
    s_j = emb_summaries[j] 
    diff = emb_orig - s_j   
    msd_j = (diff ** 2).mean() 
    global_msd.append(float(msd_j))

global_msd = np.array(global_msd)
global_msd[:10]  

array([0.00046774, 0.00045327, 0.00047498, 0.00045833, 0.00045809,
       0.00051649, 0.00043542, 0.00048906, 0.00047994, 0.00054862])

In [11]:
idx_sorted = np.argsort(global_msd)

rows = []
for rank, j in enumerate(idx_sorted, start=1):
    rows.append(
        {
            "rank": rank,
            "cluster_id": summary_cluster_ids[j],
            "summary_text": summaries[j],
            "global_msd": float(global_msd[j]),
        }
    )

df_summaries = pd.DataFrame(rows)
df_summaries.head(TOP_K_PRINCIPLES)

,rank,cluster_id,summary_text,global_msd
0,1,41,AI should consistently provide accurate and fa...,0.000414
1,2,13,"AI should interact with users in a friendly, k...",0.000435
2,3,6,"AI should provide clear, precise, and accurate...",0.000435
3,4,1,"AI should act impartially and without bias, ad...",0.000453
4,5,12,AI should be honest and clearly communicate wh...,0.000453


In [12]:
output_json_path = Path(f"consensus_principle_selection_{TRESHOLD}_fair_clustering.json")
with output_json_path.open("w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)

print("Saved JSON to:", output_json_path)

Saved JSON to: consensus_principle_selection_k=1371_fair_clustering.json


In [13]:
top = df_summaries.head(10)
for _, row in top.iterrows():
    print("Rank:", row["rank"])
    print("Cluster:", row["cluster_id"])
    print("Global MSD:", row["global_msd"])
    print("Summary:\n", row["summary_text"])
    print("-" * 80)

Rank: 1
Cluster: 41
Global MSD: 0.0004139775021167727
Summary:
 AI should consistently provide accurate and factual information.
--------------------------------------------------------------------------------
Rank: 2
Cluster: 13
Global MSD: 0.0004349993024799513
Summary:
 AI should interact with users in a friendly, kind, and respectful manner.
--------------------------------------------------------------------------------
Rank: 3
Cluster: 6
Global MSD: 0.0004354184046834607
Summary:
 AI should provide clear, precise, and accurate information in its responses.
--------------------------------------------------------------------------------
Rank: 4
Cluster: 1
Global MSD: 0.00045326992058520416
Summary:
 AI should act impartially and without bias, adhering to ethical standards.
--------------------------------------------------------------------------------
Rank: 5
Cluster: 12
Global MSD: 0.00045327072433526557
Summary:
 AI should be honest and clearly communicate when it lacks knowled

In [20]:
def select_top_k_with_similarity_aggregation(
    ranked_items,
    embed_texts,
    top_k=10,
    sim_threshold=0.85,
    text_key="summarized_principle",
    score_key="global_msd",
    ascending=True,
    id_key="cluster_id",
):
    items = [dict(x) for x in ranked_items]
    items.sort(key=lambda x: x.get(score_key, float("inf")), reverse=not ascending)

    texts = [((it.get(text_key) or "").strip()) for it in items]
    emb = np.asarray(embed_texts(texts), dtype=float)
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0
    emb = emb / norms

    selected = []

    for i, it in enumerate(items):
        if len(selected) >= top_k:
            break

        s = texts[i]
        if not s:
            continue

        if not selected:
            selected.append({
                "rank": 1,
                id_key: it.get(id_key),
                score_key: float(it.get(score_key)) if it.get(score_key) is not None else None,
                text_key: s,
                "aggregated_cluster_ids": [it.get(id_key)],
                "aggregated_count": 1,
                "aggregated_items": [{
                    id_key: it.get(id_key),
                    score_key: float(it.get(score_key)) if it.get(score_key) is not None else None,
                    text_key: s,
                }],
                "_rep_idx": i,
            })
            continue

        sims = [float(emb[i] @ emb[g["_rep_idx"]]) for g in selected]
        best_j = int(np.argmax(sims))
        best_sim = sims[best_j]

        if best_sim >= sim_threshold:
            g = selected[best_j]
            cid = it.get(id_key)
            g["aggregated_cluster_ids"].append(cid)
            g["aggregated_count"] += 1
            g["aggregated_items"].append({
                id_key: cid,
                score_key: float(it.get(score_key)) if it.get(score_key) is not None else None,
                text_key: s,
                "similarity_to_rep": best_sim,
            })
        else:
            selected.append({
                "rank": len(selected) + 1,
                id_key: it.get(id_key),
                score_key: float(it.get(score_key)) if it.get(score_key) is not None else None,
                text_key: s,
                "aggregated_cluster_ids": [it.get(id_key)],
                "aggregated_count": 1,
                "aggregated_items": [{
                    id_key: it.get(id_key),
                    score_key: float(it.get(score_key)) if it.get(score_key) is not None else None,
                    text_key: s,
                }],
                "_rep_idx": i,
            })

    for g in selected:
        g.pop("_rep_idx", None)

    return selected



In [23]:
selected_top10 = select_top_k_with_similarity_aggregation(
    ranked_items=df_summaries.to_dict("records"),
    embed_texts=embed_texts,
    top_k=10,
    sim_threshold=0.85,
    text_key="summary_text",
    score_key="global_msd",
    ascending=True,
    id_key="cluster_id",
)

for item in selected_top10:
    print("Rank:", item["rank"])
    print("Cluster:", item["cluster_id"])
    print("Global MSD:", item["global_msd"])
    print("Aggregated Count:", item["aggregated_count"])
    print("Aggregated Cluster IDs:", item["aggregated_cluster_ids"])
    print("Summary:\n", item["summary_text"])
    print("-" * 80)


Rank: 1
Cluster: 41
Global MSD: 0.0004139775021167727
Aggregated Count: 2
Aggregated Cluster IDs: [41, 6]
Summary:
 AI should consistently provide accurate and factual information.
--------------------------------------------------------------------------------
Rank: 2
Cluster: 13
Global MSD: 0.0004349993024799513
Aggregated Count: 1
Aggregated Cluster IDs: [13]
Summary:
 AI should interact with users in a friendly, kind, and respectful manner.
--------------------------------------------------------------------------------
Rank: 3
Cluster: 1
Global MSD: 0.00045326992058520416
Aggregated Count: 1
Aggregated Cluster IDs: [1]
Summary:
 AI should act impartially and without bias, adhering to ethical standards.
--------------------------------------------------------------------------------
Rank: 4
Cluster: 12
Global MSD: 0.00045327072433526557
Aggregated Count: 1
Aggregated Cluster IDs: [12]
Summary:
 AI should be honest and clearly communicate when it lacks knowledge or does not know the

In [14]:
ranked_by_size = []
for item in summarized_principles:
    ops = item.get("original_principles", [])
    cluster_size = sum(1 for p in ops if isinstance(p, (list, tuple)) and p and p[0] and str(p[0]).strip())
    ranked_by_size.append({
        "cluster_id": item.get("cluster_id"),
        "cluster_size": cluster_size,
        "summarized_principle": (item.get("summarized_principle") or "").strip(),
    })

ranked_by_size = sorted(ranked_by_size, key=lambda x: x["cluster_size"], reverse=True)

output_json_path = Path(f"size_base_principle_selection_{TRESHOLD}_fair_clustering.json")
with output_json_path.open("w", encoding="utf-8") as f:
    json.dump(ranked_by_size, f, indent=2, ensure_ascii=False)

print("Saved JSON to:", output_json_path)

pd.DataFrame(ranked_by_size[:10])

Saved JSON to: size_base_principle_selection_k=1371_fair_clustering.json


,cluster_id,cluster_size,summarized_principle
0,41,177,AI should consistently provide accurate and fa...
1,1,165,"AI should act impartially and without bias, ad..."
2,13,152,"AI should interact with users in a friendly, k..."
3,6,87,"AI should provide clear, precise, and accurate..."
4,8,69,AI should communicate in language that is clea...
5,3,58,AI should avoid providing false or harmful inf...
6,2,45,AI should not cause or encourage harm to anyone.
7,4,41,"AI should communicate in a human-like, convers..."
8,19,32,AI should tailor its responses to align with u...
9,26,27,AI should acknowledge and represent diverse pe...


In [15]:
top = pd.DataFrame(ranked_by_size[:10]).copy()
top["rank"] = range(1, len(top) + 1)

for _, row in top.iterrows():
    print("Rank:", row["rank"])
    print("Cluster:", row["cluster_id"])
    print("Cluster Size:", row["cluster_size"])
    print("Summary:\n", row["summarized_principle"])
    print("-" * 80)

Rank: 1
Cluster: 41
Cluster Size: 177
Summary:
 AI should consistently provide accurate and factual information.
--------------------------------------------------------------------------------
Rank: 2
Cluster: 1
Cluster Size: 165
Summary:
 AI should act impartially and without bias, adhering to ethical standards.
--------------------------------------------------------------------------------
Rank: 3
Cluster: 13
Cluster Size: 152
Summary:
 AI should interact with users in a friendly, kind, and respectful manner.
--------------------------------------------------------------------------------
Rank: 4
Cluster: 6
Cluster Size: 87
Summary:
 AI should provide clear, precise, and accurate information in its responses.
--------------------------------------------------------------------------------
Rank: 5
Cluster: 8
Cluster Size: 69
Summary:
 AI should communicate in language that is clear, understandable, and specific.
-----------------------------------------------------------------------

In [26]:
selected_top10_size = select_top_k_with_similarity_aggregation(
    ranked_items=ranked_by_size,
    embed_texts=embed_texts,
    top_k=10,
    sim_threshold=0.85,
    text_key="summarized_principle",
    score_key="cluster_size",
    ascending=False,
    id_key="cluster_id",
)
for item in selected_top10_size:
    print("Rank:", item["rank"])
    print("Cluster:", item["cluster_id"])
    print("Aggregated Count:", item["aggregated_count"])
    print("Aggregated Cluster IDs:", item["aggregated_cluster_ids"])
    print("Summary:\n", item["summarized_principle"])
    print("-" * 80)

Rank: 1
Cluster: 41
Aggregated Count: 2
Aggregated Cluster IDs: [41, 6]
Summary:
 AI should consistently provide accurate and factual information.
--------------------------------------------------------------------------------
Rank: 2
Cluster: 1
Aggregated Count: 1
Aggregated Cluster IDs: [1]
Summary:
 AI should act impartially and without bias, adhering to ethical standards.
--------------------------------------------------------------------------------
Rank: 3
Cluster: 13
Aggregated Count: 1
Aggregated Cluster IDs: [13]
Summary:
 AI should interact with users in a friendly, kind, and respectful manner.
--------------------------------------------------------------------------------
Rank: 4
Cluster: 8
Aggregated Count: 1
Aggregated Cluster IDs: [8]
Summary:
 AI should communicate in language that is clear, understandable, and specific.
--------------------------------------------------------------------------------
Rank: 5
Cluster: 3
Aggregated Count: 1
Aggregated Cluster IDs: [3]
S